# EPAM Armenia Jobs Scraper

**Source:** [careers.epam.com](https://careers.epam.com/en/jobs?country=4000741400000756803)  
**Filter:** Armenia (`country=4000741400000756803`)  
**Method:** Internal search API — discovered via Playwright network interception

### Scraping strategy
EPAM's careers site is a Next.js SPA. The listing page makes requests to an internal search API:  
`GET https://careers.epam.com/api/jobs/v2/search/careers-i18n`

This endpoint is same-origin but **accepts unauthenticated requests** with browser-like headers.  
Critically, each job stub already contains the **full description** (`text` field + structured `category` object with `responsibilities`, `requirements`, `nice_to_have`), so **no detail page scraping is needed**.

### API parameters
| Parameter | Value |
|---|---|
| `facets` | `country=4000741400000756803` (Armenia) |
| `size` | `30` (max per page) |
| `from` | offset (0, 30, 60, …) |
| `lang` | `en` |

### Output files
- `data/raw/jobs/epam_jobs_raw.csv` — 108 rows
- `data/processed/jobs/epam_jobs_standardized.csv` — 108 rows

In [1]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from pathlib import Path

SEARCH_URL  = "https://careers.epam.com/api/jobs/v2/search/careers-i18n"
COUNTRY_ID  = "4000741400000756803"   # Armenia
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36",
    "Referer":    "https://careers.epam.com/en/jobs",
    "Accept":     "application/json",
}

BASE_DIR = Path.cwd()
RAW_DIR  = BASE_DIR / "data" / "raw" / "jobs"
PROC_DIR = BASE_DIR / "data" / "processed" / "jobs"

print(f"API: {SEARCH_URL}")
print(f"Country filter: Armenia (id={COUNTRY_ID})")

API: https://careers.epam.com/api/jobs/v2/search/careers-i18n
Country filter: Armenia (id=4000741400000756803)


## Helper functions

In [2]:
def html_to_text(html_str):
    if not html_str: return ""
    text = BeautifulSoup(str(html_str), "html.parser").get_text("\n", strip=True)
    return re.sub(r"\n{3,}", "\n\n", text).strip()

def build_full_text(j):
    """Combine text field + structured category sections into full_text for NLP."""
    parts = []
    if j.get("text"):
        parts.append(j["text"].strip())
    cat = j.get("category") or {}
    for key in ["responsibilities", "requirements", "nice_to_have", "technologies", "about_the_project"]:
        items = cat.get(key) or []
        if items:
            label = key.replace("_", " ").title()
            parts.append(f"{label}:\n" + "\n".join(f"- {item}" for item in items))
    return "\n\n".join(parts).strip()

print("Helpers defined.")

Helpers defined.


## Step 1 — Paginate through all Armenia jobs

In [3]:
all_stubs = []
offset = 0
while True:
    params = {
        "facets":  f"country={COUNTRY_ID}",
        "from":    offset,
        "lang":    "en",
        "size":    30,
        "sortBy":  "relevance;relocation=asc",
    }
    data  = requests.get(SEARCH_URL, headers=HEADERS, params=params, timeout=20).json()
    batch = data["data"]["jobs"]
    if not batch:
        break
    all_stubs.extend(batch)
    total = data["data"]["total"]
    print(f"  Fetched {len(all_stubs)}/{total}")
    if len(all_stubs) >= total:
        break
    offset += 30
    time.sleep(0.5)

print(f"\nTotal Armenia jobs collected: {len(all_stubs)}")

  Fetched 30/108
  Fetched 60/108
  Fetched 90/108
  Fetched 108/108

Total Armenia jobs collected: 108


## Step 2 — Build records

Each job stub includes:
- `name` → job title
- `seniority` → seniority string (e.g. "Senior", "Lead")
- `job_specialization` → list of specialization names
- `primary_skill` + `skills` → structured skill tags
- `text` → plain-text description (intro paragraph)
- `category` → dict with `responsibilities`, `requirements`, `nice_to_have`, `technologies`
- `country` → list of country dicts (job may be posted across multiple countries)

In [4]:
records = []
for j in all_stubs:
    key = j.get("_key", "")
    url = f"https://careers.epam.com/en/vacancy/{key}" if key else ""

    primary = j.get("primary_skill", "") or ""
    skills_list = [primary] if primary else []
    for sk in (j.get("skills") or []):
        s = sk if isinstance(sk, str) else sk.get("name", "")
        if s and s not in skills_list:
            skills_list.append(s)
    skills_tags = ", ".join(skills_list)

    countries = [c.get("name","") if isinstance(c, dict) else str(c)
                 for c in (j.get("country") or [])]
    location  = ", ".join(countries) if countries else "Armenia"

    spec = j.get("job_specialization") or []
    department = ", ".join(spec) if isinstance(spec, list) else str(spec)

    records.append({
        "source":          "epam",
        "source_url":      url,
        "job_title":       j.get("name",""),
        "company_name":    "EPAM Systems",
        "location":        location,
        "seniority_level": j.get("seniority","") or "",
        "department":      department,
        "skills_tags":     skills_tags,
        "hiring_type":     j.get("hiring_type","") or "",
        "posting_date":    (j.get("created_at") or "")[:10],
        "full_text":       build_full_text(j),
    })

print(f"Records built: {len(records)}")
print()
for r in records[:6]:
    print(f"  {r['job_title']:55s} | {r['seniority_level']:10s} | {r['department']}")

Records built: 108

  Senior Site Reliability Engineer                        | Senior     | DevOps
  Senior ServiceNow Business Analyst                      | Senior     | Business Analyst
  Lead Test Automation Engineer in Java                   | Lead       | Automation Tester
  Key .NET Developer (EnTech domain)                      | Senior     | Developer
  Senior Functional Consultant (SAP MM / SD with Retail)  | Senior     | Other
  Senior BI Business Analyst                              | Senior     | Business Analyst


## Step 3 — Save outputs

In [5]:
df = pd.DataFrame(records)
raw_path = RAW_DIR / "epam_jobs_raw.csv"
df.to_csv(raw_path, index=False, encoding="utf-8")
print(f"Raw saved: {raw_path} ({len(df)} rows)")

std_cols = ["source","source_url","job_title","company_name","location",
            "seniority_level","department","skills_tags","posting_date","full_text"]
std_df = df[std_cols]
std_path = PROC_DIR / "epam_jobs_standardized.csv"
std_df.to_csv(std_path, index=False, encoding="utf-8")
print(f"Standardized saved: {std_path} ({len(std_df)} rows)")
print()
print("=== FIELD COVERAGE ===")
for col in std_cols:
    filled = std_df[col].replace("", pd.NA).notna().sum()
    print(f"  {col:20s}: {filled}/{len(std_df)} ({filled/len(std_df)*100:.0f}%)")
print()
ft = std_df["full_text"].str.len()
print(f"full_text — Min: {ft.min()}  Median: {ft.median():.0f}  Max: {ft.max()}")

Raw saved: /Users/lianaaghamalyan/thesis_data/data/raw/jobs/epam_jobs_raw.csv (108 rows)
Standardized saved: /Users/lianaaghamalyan/thesis_data/data/processed/jobs/epam_jobs_standardized.csv (108 rows)

=== FIELD COVERAGE ===
  source              : 108/108 (100%)
  source_url          : 108/108 (100%)
  job_title           : 108/108 (100%)
  company_name        : 108/108 (100%)
  location            : 108/108 (100%)
  seniority_level     : 108/108 (100%)
  department          : 108/108 (100%)
  skills_tags         : 108/108 (100%)
  posting_date        : 108/108 (100%)
  full_text           : 108/108 (100%)

full_text — Min: 1600  Median: 3324  Max: 7084
